In [0]:
# Installing Faker library to generate fake employee data
%pip install faker

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ---------------------------------------------------
# STEP 1: Importing required libraries
# ---------------------------------------------------
from faker import Faker
import random
import pandas as pd
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

print("Libraries imported successfully")

Libraries imported successfully


In [0]:
# ---------------------------------------------------
# STEP 2: Generating fake Employee Master data
# This will act as our Dimension table for SCD Type 2
# ---------------------------------------------------
departments = ["Sales", "HR", "Engineering", "Finance", "Marketing", "Operations"]
locations = ["Bhubaneswar", "Bangalore", "Pune", "Hyderabad", "Delhi"]

employees = []

for i in range(1, 201):  # generating 200 employees
    emp_id = f"E{1000 + i}"
    name = fake.name()
    dept = random.choice(departments)
    salary = random.randint(30000, 120000)
    manager = f"E{1000 + random.randint(1, 200)}" if i > 10 else "None"
    location = random.choice(locations)
    join_date = fake.date_between(start_date="-5y", end_date="-1y")

    employees.append({
        "emp_id": emp_id,
        "name": name,
        "department": dept,
        "salary": salary,
        "manager_id": manager,
        "location": location,
        "join_date": join_date
    })

emp_df = pd.DataFrame(employees)

print("Employee master data generated -", len(emp_df), "records")
emp_df.head()

Employee master data generated - 200 records


,emp_id,name,department,salary,manager_id,location,join_date
0,E1001,Allison Hill,HR,59256,None,Bangalore,2022-07-30
1,E1002,Megan Mcclain,Operations,43434,None,Delhi,2022-01-12
2,E1003,Allen Robinson,Sales,107397,None,Hyderabad,2023-09-15
3,E1004,Cristian Santos,Sales,33905,None,Bhubaneswar,2023-11-12
4,E1005,Kevin Pacheco,HR,60495,None,Delhi,2021-09-16


In [0]:
# ---------------------------------------------------
# STEP 3: Converting to Spark DataFrame and saving as Bronze Delta table
# ---------------------------------------------------
employees_spark_df = spark.createDataFrame(emp_df)

employees_spark_df.write.format("delta").mode("overwrite").saveAsTable("employees_bronze")

print("employees_bronze table created successfully")
print("Total rows inserted:", employees_spark_df.count())

employees_bronze table created successfully
Total rows inserted: 200


In [0]:
# ---------------------------------------------------
# STEP 4: Generating monthly Performance & Attendance data
# This will act as our Fact table for later analytics
# ---------------------------------------------------
performance_records = []

for emp in employees:
    for month in range(1, 7):  # 6 months of data
        performance_records.append({
            "emp_id": emp["emp_id"],
            "month": month,
            "year": 2026,
            "days_present": random.randint(15, 22),
            "tasks_completed": random.randint(5, 30),
            "rating": round(random.uniform(2.5, 5.0), 1)
        })

perf_df = pd.DataFrame(performance_records)

print("Performance data generated -", len(perf_df), "records")
perf_df.head()

Performance data generated - 1200 records


,emp_id,month,year,days_present,tasks_completed,rating
0,E1001,1,2026,22,19,3.6
1,E1001,2,2026,19,15,4.6
2,E1001,3,2026,16,13,4.7
3,E1001,4,2026,18,29,3.7
4,E1001,5,2026,21,15,2.6


In [0]:
# ---------------------------------------------------
# STEP 5: Saving Performance data as Bronze Delta table
# ---------------------------------------------------
performance_spark_df = spark.createDataFrame(perf_df)

performance_spark_df.write.format("delta").mode("overwrite").saveAsTable("performance_bronze")

print("performance_bronze table created successfully")
print("Total rows inserted:", performance_spark_df.count())

performance_bronze table created successfully
Total rows inserted: 1200
